<a href="https://colab.research.google.com/github/devesh-commits-ai/RAG-PDF-Assistant/blob/main/pdf_rag_assistant.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

! pip install pypdf faiss-cpu gradio groq
import os
import faiss
import gradio as gr
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
from groq import Groq

from google.colab import userdata

LLM_MODEL = "openai/gpt-oss-120b"
EMBED_MODEL = "sentence-transformers/all-MiniLM-L6-v2"

embedder = SentenceTransformer(EMBED_MODEL)
client = Groq(api_key=userdata.get("GROQ_API_KEY"))


def create_chunks(pages, chunk_size=800, overlap=150):
    chunks = []
    for page in pages:
        text, start = page["text"], 0
        while start < len(text):
            piece = text[start:start + chunk_size]
            if piece.strip():
                chunks.append({"text": piece, "page": page["page"]})
            start += chunk_size - overlap
    return chunks


def build_index(pdf_path):
    if pdf_path is None:
        return None, "Please upload a PDF."
    reader = PdfReader(pdf_path)
    pages = [
        {"page": i + 1, "text": p.extract_text() or ""}
        for i, p in enumerate(reader.pages)
    ]
    pages = [p for p in pages if p["text"].strip()]
    if not pages:
        return None, "No extractable text found (scanned PDF?)."
    chunks = create_chunks(pages)
    embs = embedder.encode(
        [c["text"] for c in chunks],
        convert_to_numpy=True,
        normalize_embeddings=True,
    ).astype("float32")

    return (chunks, embs), f"✅ Ready: {len(pages)} pages, {len(chunks)} chunks."


def ask(question, state):
    if state is None:
        return "Upload a PDF first.", ""
    if not question.strip():
        return "Type a question.", ""
    chunks, embs = state
    index = faiss.IndexFlatIP(embs.shape[1])
    index.add(embs)
    q = embedder.encode(
        [question], convert_to_numpy=True, normalize_embeddings=True
    ).astype("float32")
    scores, ids = index.search(q, 3)
    hits = [
        {**chunks[i], "score": float(s)}
        for s, i in zip(scores[0], ids[0]) if i != -1
    ]
    context = "\n\n".join(f"[Page {h['page']}]\n{h['text']}" for h in hits)

    resp = client.chat.completions.create(
        model=LLM_MODEL,
        messages=[
            {"role": "system", "content": (
                "Answer using ONLY the context provided. If the answer is not "
                "in the context, say \"I couldn't find that in the document.\" "
                "Mention page numbers when you use information.")},
            {"role": "user", "content": f"Context:\n{context}\n\nQuestion: {question}"},
        ],
        temperature=0,
        max_tokens=1000,
    )
    sources = "\n\n".join(
        f"**Page {h['page']}** (score {h['score']:.2f})\n> {h['text'][:300]}..."
        for h in hits
    )
    return resp.choices[0].message.content, "### Sources\n" + sources


with gr.Blocks(title="Chat with your PDF (RAG)") as demo:
    gr.Markdown("# 📄 Chat with your PDF\nUpload a PDF and ask questions. Answers cite page numbers.")
    state = gr.State()
    with gr.Row():
        pdf = gr.File(label="Upload PDF", file_types=[".pdf"], type="filepath")
        status = gr.Textbox(label="Status", interactive=False)
    pdf.upload(build_index, pdf, [state, status])
    question = gr.Textbox(label="Your question")
    btn = gr.Button("Ask", variant="primary")
    answer = gr.Markdown()
    sources = gr.Markdown()
    btn.click(ask, [question, state], [answer, sources])
    question.submit(ask, [question, state], [answer, sources])

demo.launch()

In [ ]:
demo.launch(share=True, debug=True)